In [1]:
import paho.mqtt.client as paho
from paho import mqtt
import base64
import time
import os
import sys

# --- Configurações OBRIGATÓRIAS ---
mqtt_broker_address = "28a6f2cda33d4edd968db56415df18b6.s1.eu.hivemq.cloud" # <<-- SEU ENDEREÇO AQUI
mqtt_port = 8883  # Porta TLS padrão do HiveMQ Cloud
mqtt_username = "vitorpq" # <<-- SEU USUÁRIO AQUI (PODE SER O MESMO OU OUTRO)
mqtt_password = "vitorpq01A" # <<-- SUA SENHA AQUI
mqtt_topic_to_subscribe = "testes/arquivos/base64/#" # <<-- Tópico para escutar (use '#' como wildcard)
# IMPORTANTE: Client ID Fixo para Sessão Persistente!
mqtt_client_id_receptor = "meu-receptor-trabalhos-01" # <<-- ESCOLHA UM ID FIXO E ÚNICO
output_directory = "arquivos_recebidos" # Pasta para salvar os PDFs
# --- Fim das Configurações ---


In [ ]:

# Criar pasta de saída se não existir
if not os.path.exists(output_directory):
    print(f"Receptor: Criando pasta de saída '{output_directory}'")
    os.makedirs(output_directory)

# Funções de Callback
def on_connect(client, userdata, flags, rc, properties=None):
    """Callback chamado quando a conexão com o broker é estabelecida."""
    if rc == 0:
        print(f"Receptor: Conectado ao Broker MQTT com Client ID '{mqtt_client_id_receptor}'!")
        # Inscrever-se no tópico APÓS conectar com sucesso
        print(f"Receptor: Inscrevendo-se no tópico '{mqtt_topic_to_subscribe}' com QoS=1...")
        client.subscribe(mqtt_topic_to_subscribe, qos=1)
    else:
        print(f"Receptor: Falha na conexão, código: {rc}")
        # Não adianta tentar inscrever se a conexão falhou

def on_subscribe(client, userdata, mid, granted_qos, properties=None):
    """Callback chamado quando a inscrição é bem-sucedida."""
    print(f"Receptor: Inscrito com sucesso! (MID: {mid}, QoS concedido: {granted_qos})")

def on_message(client, userdata, msg):
    """Callback chamado quando uma mensagem é recebida."""
    print(f"\nReceptor: Mensagem recebida!")
    print(f"  Tópico: {msg.topic}")
    print(f"  QoS: {msg.qos}")
    # print(f"  Payload (raw): {msg.payload[:50]}...") # Descomente para ver início do payload

    try:
        # 1. Decodificar payload de bytes para string UTF-8
        base64_string = msg.payload.decode('utf-8')
        print(f"  Payload (Decodificado UTF-8, primeiros 50 chars): {base64_string[:50]}...")

        # 2. Tentar decodificar a string Base64 para binário
        print("  Tentando decodificar Base64...")
        pdf_binary_content = base64.b64decode(base64_string)
        print(f"  Decodificado com sucesso! (Tamanho binário: {len(pdf_binary_content)} bytes)")

        # 3. Salvar o conteúdo binário em um arquivo PDF
        # Extrair identificador do tópico (ex: 'aluno123' de 'trabalhos/pdf/base64/aluno123')
        try:
            topic_parts = msg.topic.split('/')
            identifier = topic_parts[-1] if len(topic_parts) > 0 else "desconhecido"
        except:
            identifier = "erro_topico"

        timestamp = time.strftime("%Y%m%d_%H%M%S")
        output_filename = f"recebido_{identifier}_{timestamp}.pdf"
        output_filepath = os.path.join(output_directory, output_filename)

        print(f"  Salvando arquivo em '{output_filepath}'...")
        with open(output_filepath, 'wb') as pdf_file:
            pdf_file.write(pdf_binary_content)
        print("  Arquivo salvo com sucesso!")

    except UnicodeDecodeError:
        print("  ERRO: Payload não parece ser UTF-8 válido. Tratando como binário desconhecido.")
        # Poderia salvar o payload binário direto se necessário
        # output_filename = f"payload_binario_{time.strftime('%Y%m%d_%H%M%S')}.bin"
        # output_filepath = os.path.join(output_directory, output_filename)
        # with open(output_filepath, 'wb') as f:
        #     f.write(msg.payload)
    except base64.binascii.Error:
        print("  AVISO: Payload recebido não é Base64 válido. Pode ser outra informação (ex: metadados?).")
        print(f"  Conteúdo (como string, se possível): {msg.payload.decode('utf-8', errors='replace')}")
    except Exception as e:
        print(f"  ERRO inesperado ao processar a mensagem: {e}")

def on_disconnect(client, userdata, rc, properties=None):
    """Callback chamado quando o cliente desconecta."""
    if rc != 0:
        print(f"Receptor: Desconexão inesperada! Código: {rc}. Tentando reconectar...")
    else:
        print("Receptor: Desconectado normalmente.")

# Configuração do Cliente MQTT para o Receptor
# IMPORTANTE: clean_session=False e client_id fixo para persistência!
client = paho.Client(client_id=mqtt_client_id_receptor, clean_session=False,
                     userdata=None,
                     protocol=paho.MQTTv311)
client.on_connect = on_connect
client.on_subscribe = on_subscribe
client.on_message = on_message
client.on_disconnect = on_disconnect

# Configurar TLS (Necessário para HiveMQ Cloud)
client.tls_set(tls_version=mqtt.client.ssl.PROTOCOL_TLS)

# Configurar usuário e senha
client.username_pw_set(mqtt_username, mqtt_password)

# Definir que a sessão NÃO deve ser limpa ao conectar
# (Nota: Em Paho MQTT para v5, isso interage com Session Expiry Interval,
# mas para o comportamento básico de persistência com Paho v1.x, isso geralmente basta.
# Para Paho v2.x e MQTTv5 explícito, pode ser necessário configurar `properties` no connect)
# A forma legada (ainda funciona em muitos casos):
client._clean_session = False # Acessando atributo interno para forçar clean_session=False

# Conectar ao broker
print(f"Receptor: Conectando ao broker {mqtt_broker_address} com Client ID '{mqtt_client_id_receptor}' e clean_session=False...")
try:
    client.connect(mqtt_broker_address, mqtt_port, keepalive=60)
except Exception as e:
    print(f"Receptor: ERRO ao conectar: {e}")
    sys.exit(1)

# Iniciar o loop de rede para escutar mensagens indefinidamente
# loop_forever() gerencia reconexões automaticamente se a conexão cair
print("Receptor: Aguardando mensagens... (Pressione Ctrl+C para sair)")
try:
    client.loop_forever()
except KeyboardInterrupt:
    print("\nReceptor: Interrupção recebida. Desconectando...")
    client.disconnect()
    print("Receptor: Desconectado.")
except Exception as e:
    print(f"Receptor: Erro no loop principal: {e}")
    client.disconnect()

/var/folders/68/xhdx3jc908q7wgf4wy53rvnw0000gn/T/ipykernel_40484/1327144109.py:78: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = paho.Client(client_id=mqtt_client_id_receptor, clean_session=False,


Receptor: Conectando ao broker 28a6f2cda33d4edd968db56415df18b6.s1.eu.hivemq.cloud com Client ID 'meu-receptor-trabalhos-01' e clean_session=False...
Receptor: Aguardando mensagens... (Pressione Ctrl+C para sair)
Receptor: Conectado ao Broker MQTT com Client ID 'meu-receptor-trabalhos-01'!
Receptor: Inscrevendo-se no tópico 'testes/arquivos/base64/#' com QoS=1...
Receptor: Inscrito com sucesso! (MID: 1, QoS concedido: (1,))

Receptor: Mensagem recebida!
  Tópico: testes/arquivos/base64
  QoS: 1
  Payload (Decodificado UTF-8, primeiros 50 chars): JVBERi0xLjQNCiW0tba3DQolDQoxIDAgb2JqDQo8PA0KL0xhbm...
  Tentando decodificar Base64...
  Decodificado com sucesso! (Tamanho binário: 110998 bytes)
  Salvando arquivo em 'arquivos_recebidos/recebido_base64_20250424_175113.pdf'...
  Arquivo salvo com sucesso!
